# GPT Language Model
Реалізація мінімального GPT на базі символьного датасету (Tiny Shakespeare).  
Архітектура: Transformer з multi-head self-attention, feed-forward шарами та layer norm.

## 1. Імпорти та гіперпараметри

In [40]:
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 32       # кількість паралельних послідовностей у батчі
block_size = 128      # максимальна довжина контексту
max_iters = 2000      # кількість кроків навчання
eval_interval = 200   # інтервал оцінки лосу
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200      # кількість батчів для оцінки лосу
n_embd = 128          # розмірність ембедінгу
n_head = 4            # кількість голів уваги
n_layer = 4           # кількість трансформер-блоків
dropout = 0.2

torch.manual_seed(1337)
print(f'Використовується пристрій: {device}')

Використовується пристрій: cuda


## 2. Завантаження та підготовка даних

In [41]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-22 08:06:55--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-04-22 08:06:55 (47.9 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [42]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Розмір датасету: {len(text):,} символів')
print(f'Перші 200 символів:\n{text[:200]}')

Розмір датасету: 1,115,394 символів
Перші 200 символів:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 3. Токенізація (символьний рівень)

In [43]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'Розмір словника: {vocab_size}')
print(f'Символи: {" ".join(chars)}')

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f'Encode "hello": {encode("hello")}')
print(f'Decode назад: {decode(encode("hello"))}')

Розмір словника: 65
Символи: 
   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z
Encode "hello": [46, 43, 50, 50, 53]
Decode назад: hello


## 4. Train / Val розбивка та DataLoader

In [44]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'Train: {len(train_data):,} токенів')
print(f'Val:   {len(val_data):,} токенів')

def get_batch(split):
    """Повертає випадковий батч (x, y) для train або val."""
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size]   for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

Train: 1,003,854 токенів
Val:   111,540 токенів


## 5. Допоміжна функція оцінки лосу

In [45]:
@torch.no_grad()
def estimate_loss():
    """Усереднює лос по eval_iters батчах для train і val."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## 6. Архітектура моделі

### 6.1 Head — один головний блок self-attention

In [46]:
class Head(nn.Module):
    """Один головний блок self-attention."""

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v   = self.value(x)   # (B, T, hs)
        out = wei @ v          # (B, T, hs)
        return out

### 6.2 MultiHeadAttention — кілька голів паралельно

In [47]:
class MultiHeadAttention(nn.Module):
    """Кілька голів self-attention, конкатенованих разом."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

### 6.3 FeedForward — MLP після attention

In [48]:
class FeedFoward(nn.Module):
    """Двошарова MLP з розширенням у 4x."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

### 6.4 Block — один трансформер-блок

In [49]:
class Block(nn.Module):
    """Трансформер-блок: self-attention + feed-forward з residual та layer norm."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # residual + attention
        x = x + self.ffwd(self.ln2(x))  # residual + FFN
        return x

### 6.5 GPTLanguageModel — повна модель

In [50]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                              # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        x = tok_emb + pos_emb    # (B, T, C)
        x = self.blocks(x)       # (B, T, C)
        x = self.ln_f(x)         # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """Авторегресивна генерація нових токенів."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]                # (B, C) — лише останній крок
            probs  = F.softmax(logits, dim=-1)       # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

## 7. Ініціалізація моделі та оптимізатора

In [51]:
model = GPTLanguageModel().to(device)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Кількість параметрів: {total_params:.2f}M')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

Кількість параметрів: 0.82M


## 8. Навчання

In [ ]:
import time
start_total = time.time()
best_val_loss = float('inf')

print(f"{'='*60}")
print(f"  Старт тренування | {max_iters} кроків | пристрій: {device}")
print(f"{'='*60}\n")

for iter in range(max_iters):
    t0 = time.time()

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        elapsed = time.time() - start_total
        steps_done = iter + 1
        steps_left = max_iters - steps_done
        eta = (elapsed / steps_done) * steps_left if steps_done > 0 else 0

        pct = steps_done / max_iters
        bar_len = 30
        filled = int(bar_len * pct)
        bar = '█' * filled + '░' * (bar_len - filled)

        improved = ''
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            improved = '  ✓ новий кращий!'

        print(f"[{bar}] {pct*100:5.1f}%  крок {iter+1:>5}/{max_iters}")
        print(f"  train loss: {losses['train']:.4f}  |  val loss: {losses['val']:.4f}{improved}")
        print(f"  минуло: {elapsed:6.1f}s  |  залишилось ~{eta:.0f}s")
        print()

total_time = time.time() - start_total
print(f"{'='*60}")
print(f"  Готово! Час: {total_time:.1f}s | Кращий val loss: {best_val_loss:.4f}")
print(f"{'='*60}")

  Старт тренування | 2000 кроків | пристрій: cuda

[░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0.1%  крок     1/2000
  train loss: 3.9079  |  val loss: 3.9126  ✓ новий кращий!
  минуло:    3.5s  |  залишилось ~6968s

[███░░░░░░░░░░░░░░░░░░░░░░░░░░░]  10.1%  крок   201/2000
  train loss: 2.5037  |  val loss: 2.5013  ✓ новий кращий!
  минуло:   11.7s  |  залишилось ~105s

[██████░░░░░░░░░░░░░░░░░░░░░░░░]  20.1%  крок   401/2000
  train loss: 2.3874  |  val loss: 2.3981  ✓ новий кращий!
  минуло:   19.1s  |  залишилось ~76s

[█████████░░░░░░░░░░░░░░░░░░░░░]  30.0%  крок   601/2000
  train loss: 2.1729  |  val loss: 2.1967  ✓ новий кращий!
  минуло:   27.3s  |  залишилось ~63s

[████████████░░░░░░░░░░░░░░░░░░]  40.1%  крок   801/2000
  train loss: 1.9706  |  val loss: 2.0324  ✓ новий кращий!
  минуло:   35.5s  |  залишилось ~53s

[███████████████░░░░░░░░░░░░░░░]  50.0%  крок  1001/2000
  train loss: 1.8528  |  val loss: 1.9552  ✓ новий кращий!
  минуло:   43.1s  |  залишилось ~43s

[███████████████

## 9. Генерація тексту

In [53]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated)


Then, he king of me are you have are min Lanty
Up of his, preast dostily fraall loance myou arelven man,
And ourcle of devers you here wasodle and with the dosmes,
We do Good neelve live ame frienstations and wilge,
But tratle, with munthy bautriep,' the utmly be you
dids, I that the downs if Her my quickly
peake now is jury olojest mive of holdh yie,
A not her indry:
And Junt euntrutent my chour well.

CORIOZABELL:
Boolo ray my stindure hostame blelill the beaget with all the farn's not and me 
